# 🏆 Topic 10: Capstone - Complete End-to-End Big Data Cloud ETL Pipeline

This capstone unifies all concepts across the course into a single executable pipeline:
1. **PySpark Session Initialization & Configuration**
2. **Raw Multi-Partition Data Ingestion**
3. **Data Quality Cleaning & Schema Transformations**
4. **Window Function Aggregations & Metrics Calculation**
5. **Broadcast Hash Join Optimization**
6. **Partitioned Parquet Storage Engine Output**

---

## Hands-on Unified Capstone Execution


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
import pyspark.sql.functions as F
import os
import shutil

print("==================================================")
print("⚡ STARTING END-TO-END BIG DATA CLOUD ETL PIPELINE")
print("==================================================")

# 1. Initialize PySpark Engine
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Capstone_Big_Data_Pipeline") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("\n--- 1. Data Ingestion & Schema Definition ---")
raw_transactions = [
    ("T1001", "U101", "P_ELEC", 1200.0, "2026-08-01"),
    ("T1002", "U102", "P_CLOTH", 80.0,   "2026-08-01"),
    ("T1003", "U101", "P_ELEC", 150.0,  "2026-08-02"),
    ("T1004", "U103", "P_GROC", 45.0,   "2026-08-02"),
    ("T1005", "U102", "P_ELEC", 950.0,  "2026-08-03"),
]

df_txn = spark.createDataFrame(raw_transactions, ["txn_id", "user_id", "prod_code", "amount", "txn_date"])
df_txn.show()

# 2. Reference Dim Table for Broadcast Join
dim_products = spark.createDataFrame([
    ("P_ELEC", "Electronics", "High Value"),
    ("P_CLOTH", "Apparel", "Standard"),
    ("P_GROC", "Groceries", "Low Margin")
], ["prod_code", "category", "margin_tier"])

print("\n--- 2. Broadcast Join & Transformation ---")
df_joined = df_txn.join(F.broadcast(dim_products), on="prod_code")

# 3. Window Function Metrics
window_user = Window.partitionBy("user_id").orderBy("txn_date")
df_analytics = df_joined.withColumn("user_cumulative_spend", F.sum("amount").over(window_user)) \
                        .withColumn("txn_rank", F.row_number().over(window_user))

df_analytics.show()

# 4. Storage Partitioned Write Simulation
output_dir = "capstone_warehouse_output"
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)

print("\n--- 3. Writing Partitioned Parquet Warehouse ---")
df_analytics.write \
    .partitionBy("category") \
    .mode("overwrite") \
    .parquet(output_dir)

print(f"✅ Warehouse Parquet Files Saved to '{output_dir}/'")
print("==================================================")
print("🎉 CAPSTONE ETL PIPELINE COMPLETED SUCCESSFULLY!")
print("==================================================")
